# Higiene de demanda

La notebook 02 y `train` / `predict` usan `prepare_daily_demand`. Aquí se ve **qué cambia** respecto a agregar solo los días con ticket:

1. Calendario continuo: días sin venta → 0
2. Cap de cantidad diaria al percentil 99 (días con venta > 0)
3. Fuera el one-shot `23843` (PAPER CRAFT LITTLE BIRDIE)

No vuelve a escribir la tabla de reorden; eso lo hace la notebook 02 (o el CLI).


In [ ]:
from pathlib import Path
import sys

import pandas as pd

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "inventario_ecommerce").exists() else cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from inventario_ecommerce import config
from inventario_ecommerce.dataset import load_transactions
from inventario_ecommerce.features import (
    build_daily_sku_demand,
    clean_transactions,
    prepare_daily_demand,
)
from inventario_ecommerce.modeling.train import temporal_backtest_baseline

pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:,.2f}".format)


## 1. Sparse vs calendario

Sin rellenar ceros, rolling y media móvil solo ven días con transacción y sobreestiman la demanda diaria.


In [ ]:
raw = load_transactions()
clean = clean_transactions(raw)

sparse = build_daily_sku_demand(clean)
dense = prepare_daily_demand(clean)

print(f"Filas solo-días-con-venta: {len(sparse):,}")
print(f"Filas calendario:          {len(dense):,}")
print(f"SKUs sparse: {sparse[config.COL_STOCK_CODE].nunique():,}")
print(f"SKUs dense:  {dense[config.COL_STOCK_CODE].nunique():,}")
print(f"Outliers excluidos: {sorted(config.OUTLIER_STOCK_CODES)}")

positive = dense.loc[dense["QuantitySold"] > 0, "QuantitySold"]
print(f"Máximo diario tras winsor P{config.DAILY_QTY_WINSOR_PERCENTILE:.0%}: {positive.max():,.1f}")


## 2. Un SKU concreto

Mismo producto: a la izquierda solo hay filas cuando alguien compró; a la derecha el calendario incluye ceros.


In [ ]:
sku = "85123A"
stock = config.COL_STOCK_CODE
cols = ["Date", stock, "QuantitySold"]

sparse_sku = sparse.loc[sparse[stock].astype(str) == sku, cols]
dense_sku = dense.loc[dense[stock].astype(str) == sku, cols]

print(f"Filas sparse: {len(sparse_sku):,} | media en días con venta: {sparse_sku['QuantitySold'].mean():.2f}")
print(f"Filas dense:  {len(dense_sku):,} | media calendario:         {dense_sku['QuantitySold'].mean():.2f}")
dense_sku.head(10)


## 3. Efecto en el backtest

Mismo holdout de 30 días. El MAE “sparse” solo puntúa días con ticket (sesgado). El MAE del pipeline es el del calendario con ceros.


In [ ]:
_, sparse_global = temporal_backtest_baseline(sparse, horizon_days=30, lookback_days=30)
_, dense_global = temporal_backtest_baseline(dense, horizon_days=30, lookback_days=30)

comparison = pd.concat(
    [
        sparse_global.assign(series="solo_dias_con_venta"),
        dense_global.assign(series="calendario_con_ceros"),
    ],
    ignore_index=True,
)
comparison[["series", "GlobalMAE", "GlobalMAPE", "CutoffDate", "EvalStartDate", "EvalEndDate"]]


El MAE sobre calendario (~4 ud/día en Online Retail II) no se compara con el MAE sparse (~20): miden cosas distintas. Para inventario, la lectura correcta incluye los días a cero.
